# 00 - Install And Setup

Welcome. This notebook is a quick first check that eCAT is available in your notebook kernel and that the packaged example data can be loaded, inspected, and plotted.

The example files are small CH Instruments text exports included with the repository, so everything here should run without private lab paths. Once this works, the same pattern can be reused with your own data folder by changing `DATA_DIR`.

## Import eCAT And Locate Example Data

Start by importing eCAT and pointing the notebook at the packaged Fe/PhOH CV example folder. If `import ecat as e` fails, install eCAT into this notebook kernel first, then restart the kernel and run this cell again.

For the lab beta tag, install directly from GitHub. This requires Git to be installed and available on your `PATH`; from a terminal, `git --version` should print a version number.

Terminal install:

```bash
python -m pip install --upgrade "git+https://github.com/ljelissiry/eCAT.git@v0.1.0b4"
```

Notebook install:

```python
# %pip install --upgrade "git+https://github.com/ljelissiry/eCAT.git@v0.1.0b4"
```

The setup cell also creates a local `_outputs` folder for figures and exported data.

In [1]:
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent


def display_path(path):
    path = Path(path)
    try:
        return str(path.resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return path.name

# If this import fails, install eCAT into this notebook kernel, then restart the kernel.
# Requires Git to be installed and available on PATH.
# %pip install --upgrade "git+https://github.com/ljelissiry/eCAT.git@v0.1.0b4"
import ecat as e

DATA_DIR = ROOT / "examples" / "data" / "fe_phoh_cv"
EXPORT_DIR = ROOT / "notebooks" / "_outputs"
EXPORT_DIR.mkdir(exist_ok=True)

e.plotting_style("notebook")
print("eCAT version:", getattr(e, "__version__", "unknown"))
print("Example data:", display_path(DATA_DIR))
print("Text files:", len(list(DATA_DIR.glob("*.txt"))))

eCAT version: 0.1.0b4
Example data: examples/data/fe_phoh_cv
Text files: 13


## Discover Common Entry Points

`describe_options()` is the built-in menu for eCAT options. Calling it without a section shows the available option sections; later notebooks use section names like `plot`, `get_data`, and `multiplot`. Excel workbooks exported by eCAT can be loaded back with `get_data_from_excel()`.


In [2]:
print("Common starting points:")
print("- e.echem.from_file(path)")
print("- e.get_data({'folder path': path})")
print("- e.get_data_from_excel(path_to_workbook)")
print("- e.describe_options('plot')")

menu = e.describe_options(None, {"print": False, "return": True});
menu.head(10)


Common starting points:
- e.echem.from_file(path)
- e.get_data({'folder path': path})
- e.get_data_from_excel(path_to_workbook)
- e.describe_options('plot')


,Workflow,Function,Description
0,Overview,all,"Show every registered option table, grouped by..."
1,Import and metadata,echem.from_file,Load one supported data file and promote it to...
2,Import and metadata,get_cvs,Load CV-like text exports from one folder into...
3,Import and metadata,get_data,"Import supported electrochemistry files, parse..."
4,Import and metadata,get_data_from_excel,Load worksheet-based exported eCAT data back i...
5,Object plotting,ca.plot,"Chronoamperometry plotting options, including ..."
6,Object plotting,cp.plot,Chronopotentiometry plotting options for poten...
7,Object plotting,cv.plot,"CV trace plotting options, including segment s..."
8,Object plotting,cv.x,Options affecting CV x-axis extraction and dis...
9,Object plotting,cv.xy,Options affecting paired CV x/y extraction and...


Python's built-in `help()` is useful too. It shows the docstring for a function, including a short description and example call.

In [3]:
help(e.get_data)

Help on function get_data in module ecat.io:

get_data(options=None)
    Read electrochemistry text files from a folder into eCAT objects.

    Parameters
    ----------
    options : dict or ImportOptions, optional
        Import, parsing, sorting, and reference-shift options. See ``e.describe_options("get_data")``.

    Returns
    -------
    list of echem
        Imported CV, DPV, CA, CP, CPE, or generic electrochemistry objects.

    Examples
    --------
    >>> cvs = e.get_data({"folder path": folder, "reference mode": "keyword"})



## Load One CV With `from_file`

Use `e.echem.from_file()` when you want to inspect one text export before loading a whole folder. The returned object is saved as `cv`, and `e.show(cv)` gives a tidy Info + Stats table before any `get_data()` folder import happens.

In [4]:
first_file = DATA_DIR / "MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_to_1V_100mVs.txt"
cv = e.echem.from_file(str(first_file))

e.show(cv);

,Metric,Value
0,Name,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_to_1V_100mVs
1,Timestamp,2026-05-07 14:42:19
2,Creation Time,2026-06-08 12:16:55.081368
3,Modification Time,2026-06-08 12:16:55.082093
4,IR Comp Resistance,132 ohm
5,IR Uncomp Resistance,0 ohm
6,IR Comp Percent,100 %
7,Solvent,MeCN
8,Gas,Ar
9,Compounds,"0.1 M TBAPF6, 3 mM Fc, 1 mM Fe-tpyPY2Me"


## Load A Folder With `get_data`

Use `get_data()` when you want eCAT to read every supported text export in a folder. This returns a list of eCAT objects, one per file.

In [5]:
cvs = e.get_data({
    "folder path": str(DATA_DIR),
    "reference mode": "none",
    "print": False,
})
print(f"Loaded {len(cvs)} CV objects")
print(sorted({type(obj).__name__ for obj in cvs}))

Searching recursively through:
 examples/data/fe_phoh_cv
13 .txt files found.



Loaded 13 CV objects
['cv']


## Inspect Loaded Objects With `show_objects`

`show_objects()` gives a compact table for a list of objects. It is often the fastest way to confirm names, gases, scan rates, and compounds before filtering or plotting.

In [6]:
e.show_objects(cvs);
cvs[0].plot()

[Conditions] Exp Type: CV, Solvent: MeCN, Compounds: 0.1 M TBAPF₆


,Gas,Compounds,Scan Window,Scan Rate,Segments
[0],Ar,,"[-1.2, 1]",100 mV/s,3
[1],Ar,"3 mM Fc, 1 mM Fe-tpyPY2Me","[-1.2, 1]",100 mV/s,3
[2],Ar,"3 mM Fc, 1 mM Fe-tpyPY2Me","[-1.7, 1]",100 mV/s,3
[3],Ar,"3 mM Fc, 1 mM Fe-tpyPY2Me","[-1.7, 1]",25 mV/s,3
[4],Ar,"3 mM Fc, 1 mM Fe-tpyPY2Me","[-1.7, 1]",500 mV/s,3
[5],Ar,"3 mM Fc, 1 mM Fe-tpyPY2Me","[-1.7, 1]",1 V/s,3
[6],Ar,"3 mM Fc, 1 mM Fe-tpyPY2Me","[-1.7, 1]",50 mV/s,3
[7],CO2,"3 mM Fc, 1 mM Fe-tpyPY2Me","[-1.2, 1]",100 mV/s,3
[8],CO2,"3 mM Fc, 1 mM Fe-tpyPY2Me, 100 mM PhOH","[-1.2, 1]",100 mV/s,3
[9],CO2,"3 mM Fc, 1 mM Fe-tpyPY2Me, 560 mM PhOH","[-1.2, 1]",100 mV/s,3


<Axes: title={'center': 'MeCN, Ar, 100 mV/s, 3 seg.'}, xlabel='Potential (V)', ylabel='Current (μA)'>

## Plot The Single CV

Calling `.plot()` on an eCAT object draws the selected CV. The commented line shows the usual way to save the figure after you like how it looks.

In [7]:
ax = cv.plot()
# ax.figure.savefig(EXPORT_DIR / "first_cv.svg", dpi=300, bbox_inches="tight")